In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Candidate profile
candidate = {
    "skills": ["digital marketing"],
    "sector_interest": "marketing",
    "education": "MBA",
    "location_preference": "bangalore"
}

#  Load internship dataset
df = pd.read_csv("//content/recomm_df_education_by_category_skills.csv")
df.sample(5)

,id,job_title,company_name,job_loc,details,category,compensation,start,end,skills,href,education
519,545,us it staffing interns,isl tech solutions inc,pune,"hello,we are looking for two interns who can s...",human resources professional,paid,09-01-2017,09-06-2017,human resource practices,http://letsintern.com/internship/Human-Resourc...,B.Tech Computer Science / B.Sc Computer Science
475,498,us it staffing interns,isl tech solutions inc,anywhere in india,candidates with good communication,human resources recruiter,paid,29-03-2019,NaN,no skills preferred,http://letsintern.com/internship/Human-Resourc...,B.Tech Computer Science
542,569,marketing super - interns,district force,kolkata,we are looking for an enthusiastic mar...,business development executive,paid,28-02-2019,30-08-2019,"analytical skills,agreeableness,ad pl...",http://letsintern.com/internship/Business-Deve...,MBA (Marketing/Management)
79,81,online/ social medìa markering,the golden roses,delhi,looking for interns who can help us pl...,marketing professional,paid,14-03-2019,14-05-2019,analytical skills,http://letsintern.com/internship/Marketing-Pro...,MBA (Marketing/Management)
561,590,campus leader,bizwiz learning,new delhi,internship title “campus leader”our ca...,marketing assistant,unpaid,31-03-2019,29-09-2019,marketing,http://letsintern.com/internship/Marketing-Ass...,MBA (Marketing/Management)


# New section

In [ ]:
df['education'].value_counts()

,count
education,
B.Tech Computer Science / B.Sc Computer Science,173
B.Tech Computer Science,149
MBA (Marketing/Management),141
Bachelor's Degree,57
B.Com / Commerce,37
B.Des / Design Diploma,33
B.Ed / M.Ed,8
B.Com / M.Com,7
BBA / B.Com,7


In [ ]:


#  Step 1: Rule-based filtering (soft, not strict)
filtered = df.copy()

# Sector and location preference: not hard filter, but give bonus later
filtered["sector_match"] = filtered["category"].str.contains(candidate["sector_interest"], case=False).astype(int)
filtered["location_match"] = filtered["job_loc"].str.contains(candidate["location_preference"], case=False).astype(int)
filtered["education_match"] = filtered["education"].str.contains(candidate["education"], case=False).astype(int)

#  Step 2: Jaccard Similarity for skills
def jaccard_similarity(set1, set2):
    set1, set2 = set(set1), set(set2)
    inter = len(set1 & set2)
    union = len(set1 | set2)
    return inter / union if union > 0 else 0

scores = []
for _, row in filtered.iterrows():
    internship_skills = [s.strip() for s in str(row["skills"]).split(",")]
    score = jaccard_similarity(candidate["skills"], internship_skills)
    scores.append(score)

filtered["jaccard_score"] = scores

#  Step 3: TF-IDF fallback similarity
filtered["combined_text"] = (
    filtered["skills"].fillna('') + " " +
    filtered["category"].fillna('') + " " +
    filtered["education"].fillna('') + " " +
    filtered["job_loc"].fillna('')
)

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(filtered["combined_text"].values)

candidate_text = " ".join(candidate["skills"]) + " " + candidate["sector_interest"] + " " + candidate["location_preference"] + " " + candidate["education"]
candidate_vec = vectorizer.transform([candidate_text])

tfidf_similarities = cosine_similarity(candidate_vec, tfidf_matrix).flatten()
filtered["tfidf_score"] = tfidf_similarities

# Step 4: Hybrid Scoring
# Weights: skills (Jaccard) > sector > location > education > fallback TF-IDF
filtered["final_score"] = (
    0.5 * filtered["jaccard_score"] +
    0.2 * filtered["sector_match"] +
    0.1 * filtered["location_match"] +
    0.1 * filtered["education_match"] +
    0.1 * filtered["tfidf_score"]
)

# Step 5: Top N recommendations
top_n = filtered.sort_values(by="final_score", ascending=False).head(5)

print("\n🔹 Final Hybrid Internship Recommendations 🔹")
top_n[["job_title", "skills", "category", "job_loc","education", "final_score"]]



🔹 Final Hybrid Internship Recommendations 🔹


,job_title,skills,category,job_loc,education,final_score
2,digital marketing internship,digital marketing,marketing professional,bangalore,MBA (Marketing/Management),0.989876
411,digital marketing & social media marketing intern,digital marketing,marketing assistant,"bangalore,anywhere in india pune",MBA (Marketing/Management),0.977207
547,social marketing,digital marketing,marketing professional,anywhere in india,MBA (Marketing/Management),0.877200
352,digital marketing internship,digital marketing,digital marketing professional,anywhere in india,B.Tech Computer Science,0.767858
44,digital marketing intern,digital marketing,digital marketing manager,mumbai,B.Tech Computer Science,0.767248
